# Vector Databases for RAG — A Deeper Dive

This notebook is built to go further than a typical intro course: instead of just calling
a vector DB API, you'll build the intuition from the ground up — what an embedding actually
is, why cosine similarity works, how approximate nearest-neighbor search scales to millions
of vectors, and how all of this plugs into a real RAG pipeline.

**Stack we'll use:**
- `sentence-transformers` — free, local embedding model (Anthropic doesn't have an embeddings API, this is the standard real-world pairing)
- `chromadb` — our vector database engine (Part 4 only, to show its raw HNSW internals)
- `langchain_chroma` — the LangChain wrapper around Chroma we use everywhere else, for a generalizable `VectorStore`/`Document` interface
- `langchain_anthropic` — Claude API for the generation half of RAG, via LangChain's chat model interface
- `numpy` / `matplotlib` — for the "under the hood" math and visualizations

**Structure:**
1. What is an embedding, really?
2. Distance metrics from scratch
3. Visualizing embedding space
4. How ANN search works (and why you need it)
5. Chroma fundamentals
6. Chunking strategies and why they matter
7. Full RAG pipeline (Chroma + Claude)
8. Evaluating retrieval quality
9. Advanced: hybrid search, re-ranking, metadata filtering
10. Exercises

Run the setup cell below first.


In [1]:
# Setup — run this first
# !pip install -q sentence-transformers chromadb anthropic numpy matplotlib scikit-learn rank_bm25 langchain-chroma langchain-huggingface langchain-anthropic langchain-core

import numpy as np
# import matplotlib.pyplot as plt
print("Setup complete.")


Setup complete.


## 1. What is an embedding, really?

An embedding is just a list of numbers (a vector) that represents the *meaning* of a piece
of text. Text that means similar things ends up as vectors that point in similar directions
in a high-dimensional space.

The model doesn't store meaning explicitly — it learned, during training, to place text with
similar context near each other. "king" and "queen" end up close together; "king" and
"banana" end up far apart.

Let's load a real embedding model and see this in action.


In [2]:

from sentence_transformers import SentenceTransformer

# A small, fast, genuinely good embedding model (384 dimensions)
model = SentenceTransformer('all-MiniLM-L6-v2')

words = ["king", "queen", "man", "woman", "banana", "fruit", "castle"]
embeddings = model.encode(words)

print(f"Shape of embeddings: {embeddings.shape}")  # (7 words, 384 dimensions each)
print(f"First 10 values for 'king':\n{embeddings[0][:10]}")


/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/upskill2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape of embeddings: (7, 384)
First 10 values for 'king':
[-0.0595993   0.05051239 -0.06951004  0.07968025 -0.04674774  0.00098882
  0.07904328 -0.01273938  0.05839579 -0.03140248]


Notice: 384 numbers to represent a single word. Each dimension doesn't mean anything
human-interpretable on its own (unlike, say, a spreadsheet column called "price"). Meaning is
encoded in the *pattern* across all 384 numbers, and in how that pattern relates to other
vectors' patterns.

The famous example: `king - man + woman ≈ queen`. Let's actually test that.


In [3]:
import numpy as np
king, man, woman, queen = embeddings[0], embeddings[2], embeddings[3], embeddings[1]

result_vector = king - man + woman

# Compare result_vector to 'queen' using cosine similarity (defined properly in Part 2)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"similarity(king - man + woman, queen) = {cosine_sim(result_vector, queen):.4f}")
print(f"similarity(king - man + woman, banana) = {cosine_sim(result_vector, embeddings[4]):.4f}")


similarity(king - man + woman, queen) = 0.5795
similarity(king - man + woman, banana) = 0.3036


The analogy holds up numerically — the arithmetic on the vectors mirrors the semantic
relationship. This is the entire foundation RAG is built on: **turn text into vectors, and
"similar meaning" becomes "nearby vectors."**


## 2. Distance metrics from scratch

A vector DB's core job is: given a query vector, find the closest stored vectors. "Closest"
needs a precise definition. There are three you'll see constantly:

- **Cosine similarity** — measures the *angle* between vectors, ignoring magnitude. Ranges -1 to 1 (1 = identical direction).
- **Euclidean distance (L2)** — straight-line distance between two points. 0 = identical, larger = further apart.
- **Dot product** — like cosine similarity but *does* care about magnitude (vector length).

Most text embedding models (including MiniLM above) are trained so that **cosine similarity**
is the meaningful metric — magnitude carries little/no semantic information for these models.

Let's implement all three manually (no library shortcuts) so you see exactly what's happening.


In [4]:
def cosine_similarity_manual(a, b):
    dot_product = np.sum(a * b)
    norm_a = np.sqrt(np.sum(a ** 2))
    norm_b = np.sqrt(np.sum(b ** 2))
    return dot_product / (norm_a * norm_b)

def euclidean_distance_manual(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def dot_product_manual(a, b):
    return np.sum(a * b)

# Sanity check against numpy/scipy built-ins
from scipy.spatial.distance import cosine as scipy_cosine_dist

a, b = embeddings[0], embeddings[1]  # king, queen

print("Cosine similarity  :", cosine_similarity_manual(a, b))
print("  (scipy check, 1 - cosine distance):", 1 - scipy_cosine_dist(a, b))
print("Euclidean distance :", euclidean_distance_manual(a, b))
print("Dot product         :", dot_product_manual(a, b))


Cosine similarity  : 0.6807127
  (scipy check, 1 - cosine distance): 0.6807126402854919
Euclidean distance : 0.7991087
Dot product         : 0.6807127


**Why does cosine similarity ignore magnitude, and why does that matter for text?**

A long document and a short document about the exact same topic can produce vectors pointing
the same direction but with very different lengths (magnitude often correlates with text
length / token count for some encoders). Cosine similarity says "these mean the same thing,"
which is usually what you want. Euclidean distance would be thrown off by the length
difference. This is why virtually every vector DB defaults to cosine similarity for text
embeddings.


## 3. Visualizing embedding space

384 dimensions is impossible to picture directly, but we can compress it down to 2D with
PCA (Principal Component Analysis) just to build intuition. You'll lose information doing
this, so never use this for actual retrieval — it's purely for visualization.


In [5]:
# from sklearn.decomposition import PCA

# concepts = ["dog", "cat", "puppy", "kitten", "car", "truck", "vehicle",
#             "happy", "joyful", "sad", "depressed", "python", "javascript", "programming"]
# concept_embeddings = model.encode(concepts)

# pca = PCA(n_components=2)
# reduced = pca.fit_transform(concept_embeddings)

# plt.figure(figsize=(9, 7))
# plt.scatter(reduced[:, 0], reduced[:, 1])
# for i, word in enumerate(concepts):
#     plt.annotate(word, (reduced[i, 0], reduced[i, 1]), fontsize=11)
# plt.title("2D projection of embedding space (via PCA)")
# plt.xlabel("Principal Component 1")
# plt.ylabel("Principal Component 2")
# plt.grid(alpha=0.3)
# plt.show()


You should see rough clusters: animals near each other, vehicles near each other,
emotions near each other, programming languages near each other. That clustering *is* what
makes semantic search possible — a query about "kitten" will land near "cat" in this space
even if the word "cat" never appears in the query.


## 4. How does ANN (Approximate Nearest Neighbor) search actually work?

im a bit confused.. everything weve been talking about so far is for word embeddings but isnt rag vector databases sentence embeddings?

Here's the problem: if you have 10 million documents, comparing a query vector against all
10 million with cosine similarity ("brute force" / "exhaustive search") is $O(n)$ — it gets
slow. Vector databases solve this using **Approximate Nearest Neighbor (ANN)** algorithms,
which trade a small amount of accuracy for massive speed gains.

Chroma (like most modern vector DBs) uses **HNSW — Hierarchical Navigable Small World
graphs** under the hood. The intuition:

- Vectors are organized into a multi-layer graph, like a skip-list.
- The top layer has very few nodes with long-range connections ("highways").
- Each layer down has more nodes with shorter-range connections ("side streets").
- A search starts at the top layer, jumps toward the query's neighborhood quickly, then
  descends layer by layer, refining the search locally.
- This gets you from "check 10 million vectors" to "check a few hundred," at the cost of
  occasionally missing the *exact* single best match (hence "approximate").

Let's demonstrate the speed difference with brute force vs. a simple approximate approach,
using a synthetic dataset large enough to show the effect.


In [6]:
import time

# Simulate a modestly large dataset of random 384-dim vectors (normalized, like real embeddings)
np.random.seed(42)
n_vectors = 50_000
dim = 384

dataset = np.random.randn(n_vectors, dim).astype(np.float32)
dataset /= np.linalg.norm(dataset, axis=1, keepdims=True)  # normalize to unit vectors

query = np.random.randn(dim).astype(np.float32)
query /= np.linalg.norm(query)

# --- Brute force: compare against every single vector ---
start = time.time()
similarities = dataset @ query  # cosine similarity, since everything is unit-normalized
top_5_brute = np.argsort(-similarities)[:5]
brute_force_time = time.time() - start

print(f"Brute force search over {n_vectors:,} vectors took {brute_force_time*1000:.2f} ms")
print(f"Top 5 indices: {top_5_brute}")


Brute force search over 50,000 vectors took 7.26 ms
Top 5 indices: [21034 19141  7695 27089 41983]


In [7]:
# # Now let's see how Chroma (using HNSW internally) handles the same scale.
# # NOTE: this cell intentionally stays on raw `chromadb` rather than `langchain_chroma` —
# # we're adding pre-computed random vectors with no source text, which is exactly the
# # case langchain_chroma isn't built for (it's a Document/text-oriented VectorStore wrapper).
# import chromadb

# client = chromadb.Client()  # in-memory, ephemeral — fine for this demo
# collection = client.create_collection(name="ann_demo1", metadata={"hnsw:space": "cosine"})

# # Chroma wants string IDs and expects embeddings as lists
# ids = [str(i) for i in range(n_vectors)]
# batch_size = 5000  # must be <= 5461

# for i in range(0, n_vectors, batch_size):
#     collection.add(
#         ids=ids[i:i+batch_size],
#         embeddings=dataset[i:i+batch_size].tolist()
#     )

# start = time.time()
# results = collection.query(query_embeddings=[query.tolist()], n_results=5)
# hnsw_time = time.time() - start

# print(f"Chroma (HNSW) search over {n_vectors:,} vectors took {hnsw_time*1000:.2f} ms")
# print(f"Top 5 indices: {results['ids'][0]}")
# print(f"\nSpeedup: {brute_force_time / hnsw_time:.1f}x faster")
# print(f"Overlap with brute-force top 5: {len(set(top_5_brute.astype(str)) & set(results['ids'][0]))}/5 matched")


At 50k vectors the difference is already noticeable; at 10M+ vectors (a real production
scale) brute force becomes unusable while HNSW stays fast — that overlap count shows the
"approximate" part: HNSW is *usually* exact for top results at this scale but isn't
mathematically guaranteed to be, which is the tradeoff you're accepting for the speed.


## 5. Chroma fundamentals

Now the practical side. Chroma's core concepts:

- **Collection** — like a table; holds documents + their embeddings + metadata.
- **Documents** — the actual text (Chroma can auto-embed these for you, or you can pass your own vectors).
- **Metadata** — arbitrary key/value tags per document (source, date, category...) — used for filtering.
- **IDs** — unique string identifier per entry.

Let's build a small real collection from scratch.


In [8]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# LangChain's Embeddings interface, wrapping the same sentence-transformers model
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

documents = [
    "Python was created by Guido van Rossum and first released in 1991.",
    "JavaScript was designed in just 10 days by Brendan Eich in 1995.",
    "The Eiffel Tower was completed in 1889 for the World's Fair in Paris.",
    "Mount Everest stands at 8,849 meters, the tallest mountain above sea level.",
    "Rust is a systems programming language focused on memory safety without garbage collection.",
    "The Great Wall of China stretches over 21,000 kilometers.",
    "Go (Golang) was created at Google and released publicly in 2009.",
]

# langchain_chroma works with Document objects: page_content + metadata, instead of
# chromadb's parallel documents/metadatas/ids lists
docs = [
    Document(
        page_content=text,
        metadata={
            "category": "programming" if i in [0, 1, 4, 6] else "landmarks",
            "doc_id": f"doc_{i}",  # kept in metadata so we can recover the original id after search
        },
    )
    for i, text in enumerate(documents)
]

collection2 = Chroma.from_documents(
    documents=docs,
    embedding=hf_embeddings,
    collection_name="tech_facts",
    ids=[f"doc_{i}" for i in range(len(documents))],
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Collection now has {len(collection2.get()['ids'])} documents")


Collection now has 7 documents


why dont you need to convgert it to a retriever for this to work?

In [9]:
# Query it — similarity_search_with_score returns (Document, distance) pairs
results = collection2.similarity_search_with_score(
    "What programming language is good for memory-safe systems code?",
    k=2,
)

for doc, dist in results:
    print(f"[distance={dist:.4f}] {doc.page_content}")


[distance=0.3277] Rust is a systems programming language focused on memory safety without garbage collection.
[distance=0.7825] Python was created by Guido van Rossum and first released in 1991.


In [10]:
print(results)

[(Document(id='doc_4', metadata={'category': 'programming', 'doc_id': 'doc_4'}, page_content='Rust is a systems programming language focused on memory safety without garbage collection.'), 0.3276791572570801), (Document(id='doc_0', metadata={'doc_id': 'doc_0', 'category': 'programming'}, page_content='Python was created by Guido van Rossum and first released in 1991.'), 0.782503604888916)]


In [11]:
# Metadata filtering — restrict the search space before/while doing similarity search
results_filtered = collection2.similarity_search_with_score(
    "Tell me something old and impressive",
    k=2,
    filter={"category": "landmarks"},  # langchain_chroma's `filter` maps to chromadb's `where`
)

for doc, dist in results_filtered:
    print(f"[distance={dist:.4f}] {doc.page_content}")


[distance=0.9182] Mount Everest stands at 8,849 meters, the tallest mountain above sea level.
[distance=0.9376] The Great Wall of China stretches over 21,000 kilometers.


Note the `distance` values here are cosine **distance** (lower = more similar), which is
`1 - cosine_similarity`. Different libraries report this differently (similarity vs.
distance) — always check the docs for whichever DB you use so you sort in the right
direction.


## 6. Chunking strategies — and why they change your results

Real documents are too long to embed as one vector (you'd lose granularity — a 10-page
document about five different topics gets flattened into one blurry average vector). So you
split ("chunk") documents before embedding. **How** you chunk has a big effect on retrieval
quality.

Three common strategies:
1. **Fixed-size chunking** — split every N characters/tokens, often with overlap. Simple, but can cut a sentence (or an idea) in half.
2. **Recursive character splitting** — try to split on paragraph breaks first, then sentences, then words, only falling back to hard cuts if needed. (This is what LangChain's `RecursiveCharacterTextSplitter` does — you've used this before.)
3. **Semantic chunking** — embed sentences, then group sentences together only while consecutive sentences stay similar to each other; start a new chunk when the topic shifts.

Let's build a small example that shows *why* naive fixed-size chunking can hurt.


In [12]:
sample_text = (
    "The Amazon rainforest produces about 20 percent of the world's oxygen and is home to "
    "millions of species of plants and animals. It spans nine countries in South America. "
    "In contrast, the Sahara Desert is the largest hot desert in the world, covering much of "
    "North Africa. Temperatures there can exceed 50 degrees Celsius during the day. "
    "Meanwhile, the Python programming language emphasizes readability through its use of "
    "significant whitespace, and it has become one of the most popular languages for data science."
)

# Naive fixed-size chunking, no regard for sentence boundaries
def fixed_chunk(text, size=80):
    return [text[i:i+size] for i in range(0, len(text), size)]

bad_chunks = fixed_chunk(sample_text, size=80)
for i, c in enumerate(bad_chunks):
    print(f"Chunk {i}: {c!r}")


Chunk 0: "The Amazon rainforest produces about 20 percent of the world's oxygen and is hom"
Chunk 1: 'e to millions of species of plants and animals. It spans nine countries in South'
Chunk 2: ' America. In contrast, the Sahara Desert is the largest hot desert in the world,'
Chunk 3: ' covering much of North Africa. Temperatures there can exceed 50 degrees Celsius'
Chunk 4: ' during the day. Meanwhile, the Python programming language emphasizes readabili'
Chunk 5: 'ty through its use of significant whitespace, and it has become one of the most '
Chunk 6: 'popular languages for data science.'


Notice how fixed-size chunking slices straight through sentences and even across topics
(rainforest / desert / Python all bleeding into each other). An embedding of one of these
chunks will represent a muddled mix of unrelated ideas.

Now let's do it properly with sentence-aware (recursive-style) chunking.


In [13]:
import re

def sentence_aware_chunk(text, max_chars=150):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) <= max_chars:
            current += (" " if current else "") + sentence
        else:
            if current:
                chunks.append(current)
            current = sentence
    if current:
        chunks.append(current)
    return chunks

good_chunks = sentence_aware_chunk(sample_text, max_chars=150)
for i, c in enumerate(good_chunks):
    print(f"Chunk {i}: {c!r}")


Chunk 0: "The Amazon rainforest produces about 20 percent of the world's oxygen and is home to millions of species of plants and animals."
Chunk 1: 'It spans nine countries in South America. In contrast, the Sahara Desert is the largest hot desert in the world, covering much of North Africa.'
Chunk 2: 'Temperatures there can exceed 50 degrees Celsius during the day.'
Chunk 3: 'Meanwhile, the Python programming language emphasizes readability through its use of significant whitespace, and it has become one of the most popular languages for data science.'


Each chunk here stays topically coherent — no mid-sentence cuts, and (in this example)
each chunk roughly matches one topic. In a real project, `langchain.text_splitter.RecursiveCharacterTextSplitter`
does this same job with more configurability (chunk size, overlap, custom separators), and
for high-stakes RAG you can go further with true semantic chunking (embedding each sentence
and grouping by similarity) — worth trying as an exercise at the end of this notebook.


## 7. Full RAG pipeline: Chroma + Claude

Now let's put it all together: chunk documents → embed with sentence-transformers → store
in Chroma → retrieve relevant chunks for a question → pass them to Claude as context →
get a grounded answer.

**You'll need your Anthropic API key.** Set it as an environment variable so it's never
hardcoded in the notebook:

```bash
export ANTHROPIC_API_KEY="your-key-here"
```


In [14]:
import os
from langchain_anthropic import ChatAnthropic

# Reads from your shell environment — never hardcode a key in a notebook you might commit/push
api_key = os.environ.get("ANTHROPIC_API_KEY")
assert api_key, "Set your ANTHROPIC_API_KEY environment variable."

claude = ChatAnthropic(model="claude-sonnet-4-5", api_key=api_key, max_tokens=400)
print("Anthropic client ready (via langchain_anthropic).")


AssertionError: Set your ANTHROPIC_API_KEY environment variable.

why dont we use chroma.from_documents like I have in other examples?

In [ ]:
# A small original knowledge base for the demo (feel free to swap in your own documents)
knowledge_base = [
    "Chroma stores embeddings using an HNSW index by default, which trades perfect accuracy for large speed gains at scale.",
    "Cosine similarity is the standard metric for comparing text embeddings because it ignores vector magnitude and focuses on direction.",
    "Chunk overlap (typically 10-20% of chunk size) helps prevent losing context at chunk boundaries during retrieval.",
    "RAG (Retrieval-Augmented Generation) reduces hallucination by grounding an LLM's answer in retrieved source documents rather than relying purely on parametric memory.",
    "Re-ranking is a second-pass step where a slower, more accurate model re-scores the top-k retrieved candidates from the fast vector search.",
    "Metadata filtering lets you narrow a vector search to a subset of documents (e.g., only 2024 documents) before or during the similarity search itself.",
]

kb_docs = [
    Document(page_content=text, metadata={"doc_id": f"kb_{i}"})
    for i, text in enumerate(knowledge_base)
]

rag_collection = Chroma.from_documents(
    documents=kb_docs,
    embedding=hf_embeddings,
    collection_name="rag_demo",
    ids=[f"kb_{i}" for i in range(len(knowledge_base))],
    collection_metadata={"hnsw:space": "cosine"},
)
print("Knowledge base indexed.")


In [ ]:
def answer_with_rag(question, n_context=3):
    # 1. Retrieve
    retrieved_docs = rag_collection.similarity_search(question, k=n_context)
    context_chunks = [doc.page_content for doc in retrieved_docs]
    context_text = "\n".join(f"- {c}" for c in context_chunks)

    # 2. Augment the prompt with retrieved context
    prompt = f"""Answer the question using ONLY the context below. If the context doesn't
contain the answer, say so explicitly rather than guessing.

Context:
{context_text}

Question: {question}"""

    # 3. Generate — langchain_anthropic's ChatAnthropic uses the standard .invoke() interface
    response = claude.invoke(prompt)

    print("Retrieved context:")
    print(context_text)
    print("\n--- Claude's answer ---")
    print(response.content)

answer_with_rag("Why would I use re-ranking on top of vector search?")


Done — rather than bolting on a separate LangChain cell, the retrieval + generation cells
above were rewritten in place to use `langchain_chroma.Chroma` and `langchain_anthropic.ChatAnthropic`
throughout, so the whole notebook is now consistent (and swappable to a different vector
store or model provider later without touching the pipeline logic).

Try changing the question to something the knowledge base *doesn't* cover (e.g., "What's
the capital of France?") — a well-prompted RAG system should say it doesn't know, rather than
letting Claude's own trained knowledge answer un-grounded. That distinction — grounded vs.
ungrounded answers — is the entire point of RAG.


## 8. Evaluating retrieval quality

It's easy to build a RAG pipeline that *looks* like it works. Actually measuring retrieval
quality requires a labeled evaluation set: for each test question, you know in advance which
document(s) are the correct/relevant ones.

Standard retrieval metrics:

- **Precision@k** — of the top k retrieved results, what fraction are actually relevant?
- **Recall@k** — of all relevant documents that exist, what fraction did we find in the top k?
- **MRR (Mean Reciprocal Rank)** — for the *first* relevant result, how high up was it ranked? (1/rank, averaged across queries)

Let's build a tiny labeled eval set against our knowledge base and compute these.


In [2]:
# (question, set of relevant doc_ids) — ground truth we define ourselves
eval_set = [
    ("How does Chroma make search fast?", {"kb_0"}),
    ("What similarity metric should I use for text embeddings?", {"kb_1"}),
    ("How do I stop an LLM from making things up?", {"kb_3"}),
    ("How can I improve retrieved results after the initial search?", {"kb_4"}),
]

def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / len(relevant_ids)

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0.0

k = 3
precisions, recalls, rr_scores = [], [], []

for question, relevant in eval_set:
    result_docs = rag_collection.similarity_search(question, k=k)
    # doc_id was stashed in metadata when we indexed, since langchain's Document
    # doesn't otherwise expose the underlying chroma id
    retrieved_ids = [doc.metadata["doc_id"] for doc in result_docs]

    p = precision_at_k(retrieved_ids, relevant, k)
    r = recall_at_k(retrieved_ids, relevant, k)
    rr = reciprocal_rank(retrieved_ids, relevant)

    precisions.append(p)
    recalls.append(r)
    rr_scores.append(rr)

    print(f"Q: {question}")
    print(f"  Retrieved: {retrieved_ids} | Precision@{k}={p:.2f} Recall@{k}={r:.2f} RR={rr:.2f}\n")

print(f"Mean Precision@{k}: {np.mean(precisions):.3f}")
print(f"Mean Recall@{k}:    {np.mean(recalls):.3f}")
print(f"MRR:                {np.mean(rr_scores):.3f}")


NameError: name 'rag_collection' is not defined

In a real project, you'd build an eval set like this from actual user queries + human
judgment of relevant sources, and re-run it every time you change your chunking strategy,
embedding model, or `k` value — that's how you know if a change actually helped instead of
just guessing.


## 9. Advanced: hybrid search & re-ranking

Two upgrades that show up constantly in production RAG systems:

**Hybrid search** — pure vector (semantic) search can miss exact keyword matches (e.g., an
exact product code, a rare proper noun, an acronym) because embeddings prioritize *meaning*
over *exact tokens*. Hybrid search combines vector similarity with a classic keyword method
like **BM25**, then merges/re-scores the two result sets.

**Re-ranking** — vector search is fast but approximate; a re-ranker (often a cross-encoder
model) is slower but far more precise. The typical pattern: retrieve top ~20-50 candidates
fast with vector search, then re-rank just those with a heavier model, and keep the top 3-5.

Let's demonstrate hybrid search with BM25 + our vector search.


In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize the knowledge base for BM25 (simple whitespace tokenization for this demo)
tokenized_corpus = [doc.lower().split() for doc in knowledge_base]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(question, k=3, alpha=0.5):
    # Vector search scores (convert distance -> similarity)
    vec_results = rag_collection.similarity_search_with_score(question, k=len(knowledge_base))
    vec_scores = {
        doc.metadata["doc_id"]: 1 - dist
        for doc, dist in vec_results
    }

    # BM25 keyword scores
    tokenized_query = question.lower().split()
    bm25_raw_scores = bm25.get_scores(tokenized_query)
    # normalize BM25 scores to 0-1 for fair blending
    max_bm25 = max(bm25_raw_scores) if max(bm25_raw_scores) > 0 else 1
    bm25_scores = {f"kb_{i}": score / max_bm25 for i, score in enumerate(bm25_raw_scores)}

    # Blend: alpha weights vector search, (1-alpha) weights keyword search
    combined = {
        doc_id: alpha * vec_scores.get(doc_id, 0) + (1 - alpha) * bm25_scores.get(doc_id, 0)
        for doc_id in vec_scores
    }
    ranked = sorted(combined.items(), key=lambda x: -x[1])[:k]
    return ranked

results = hybrid_search("HNSW index speed tradeoff")
for doc_id, score in results:
    idx = int(doc_id.split("_")[1])
    print(f"[{score:.3f}] {knowledge_base[idx]}")


Try tuning `alpha` — closer to 1.0 leans on semantic search, closer to 0.0 leans on
keyword overlap. In production systems (e.g. Weaviate, Pinecone's hybrid mode, Elasticsearch
with vector plugins) this blending is often built in, but knowing what's happening underneath
means you can debug it when results look wrong.


## 10. Exercises — build understanding by extending this

Try these yourself to cement what you've learned (no solutions given — that's the point):

1. **Swap the embedding model.** Try `all-mpnet-base-v2` (slower, higher quality) instead of
   `all-MiniLM-L6-v2` and re-run the evaluation section. Does precision/recall change?
2. **Break chunking on purpose.** Feed a real multi-topic document (a Wikipedia article, your
   own notes) through both `fixed_chunk` and `sentence_aware_chunk`, embed both chunk sets,
   and query with a question — compare which chunking strategy retrieves the right chunk.
3. **Implement semantic chunking.** Embed each sentence individually, then group consecutive
   sentences into a chunk only while cosine similarity to the previous sentence stays above
   a threshold (e.g. 0.5). Start a new chunk when it drops below.
4. **Persist your Chroma collection.** So far we used the ephemeral in-memory client. Pass
   `persist_directory=...` to `Chroma.from_documents(...)` (or construct `Chroma(persist_directory=...)`)
   and rebuild the RAG pipeline so it survives a kernel restart.
5. **Add a real re-ranker.** Install `sentence-transformers` cross-encoder models (e.g.
   `cross-encoder/ms-marco-MiniLM-L-6-v2`) and use one to re-score the top 10 hybrid search
   results from Part 9, keeping only the top 3.
6. **Break HNSW's approximation.** Increase `n_vectors` in Part 4 to 500,000 and decrease
   Chroma's `hnsw:construction_ef` / `hnsw:search_ef` parameters (check Chroma docs for exact
   names) — watch precision vs. brute force drop as you push the approximation harder.

If you get through all six, you'll understand this material more deeply than most people who
finish a full Coursera specialization on the topic.
